<!-- Notes:

Read this and follow the steps to organize and stuff.
https://medium.com/@caneuenschwander/how-to-turn-a-messy-jupyter-notebook-into-a-professional-python-project-f34d5ee7f88b



-->

[![PythonAnywhere](https://img.shields.io/badge/Python-3776AB?logo=python&logoColor=fff)](https://www.pythonanywhere.com/user/dongwkim/)
[![Anaconda](https://img.shields.io/badge/Anaconda-44A833?logo=anaconda&logoColor=fff)](https://anaconda.com/app/)
[![JupyterLite](https://img.shields.io/badge/Jupyter-ffa500?logo=Jupyter&logoColor=fff)](https://dong-wkim.github.io/jupyterlite/lab/index.html)

<!-- Target -->
<a id="titlepage"></a>

<!-- 1. University Name (English) -->
<div align = "center" style = "font-family: Times New Roman; font-variant: small-caps;"><h5>Jagiellonian University<br>Medical College<br>Doctoral School of Medical and Health Sciences</h5></div></br>

<!-- 2. Logo -->
<div align = "center"><a href="https://dongwkim.com"><img src="https://raw.githubusercontent.com/dong-wkim/assets/refs/heads/master/logos/uj_logo.jpg" alt="Logo" width="100" height="100"></a></div>
<hr style = "background-color:gray; height: 1.5px;">
    
<!-- 3. Title -->
<div align="center" style="font-family:Times New Roman;"><h2>Comparative analysis of outcome measures using<br>the most commonly used graft harvesting sources in<br> primary anterior cruciate ligament reconstruction surgery. </h2></div>

<!-- 4. Subtitle -->
<div align="center" style="font-family:Times New Roman;"><h4><i>A systematic review and network meta-analysis<i></h4></div>
<hr style = "background-color:black; height: 1.5px;">

<div align="center" style="font-family:Times New Roman;">
    <h6>A dissertation submitted in partial fulfillment of the requirements for the award of the degree of:</h6><h5>Doctor of Philosophy (Ph.D.)</h5><h6><i>submitted by:</i></h6><br>
</div>

<!-- 5. Author -->
<div align = "center" style="font-family:Times New Roman;">
    <h4 size = 20><b>Dong Woon Kim, M.D.</b><sup>1,2,3</sup></h4>
    <h6><i>dr n. med. Dong Woon Kim</i></h6></div><br>

<!-- 6. Supervisor -->
<div align = "center"><h6 style="font-family:Times New Roman;"><i>supervised by:</i></sub></div>
<div align = "center"><sub style="font-family:Times New Roman;">Konrad Malinowski, M.D., Ph.D<sup>1,4</sup></sub></div>
</div>
<br>
<br>
<div align = "center" style = "font-family: Times New Roman;"><h6>Kraków, 2026</h6></div>
<div>

<div align = "left" style = "font-family: Times New Roman;">

<p>_________________________________________</p>
    <font family = "Times New Roman" size = "1em">
        <sup>1</sup> Department of Anatomy, Jagiellonian University, Kraków, Poland  <br/>
        <sup>2</sup> Whiting College of Engineering, Johns Hopkins University, Baltimore, MD, United States  <br/>
        <sup>3</sup> Harvard University, Cambridge, MA, United States<br>
        <sup>4</sup> Artromedical Orthopedic Clinic, Bełchatów, Poland  
    </font>
</div>

<a id="toc"></a>

- [Systematic Review](#systematic_review)
- [Data Collection](#data_collection)
- [Network Meta-Analysis](#network_meta_analysis)

<!---
Put all supportive scripts into an __init__.py file or main.py file (?)
This README file must be now deconstructured, picked apart piece by piece until you're left with:

1. back-end scripts are placed in src folder as py files that are imported.
2. individual ipynb notebooks (for testing and easy conversion) that are converted into markdown -> ReStructuredText and MyST-MD, which are deployed to
3. Sphinx OR jupyter book with interactive widgets that is hosted / deployed to the web
4. the documentation remains

--->

In [26]:
# shouldn't require internet 

requirements = [
    "pandas",
    "biopython",
    "ipython",
    "ipywidgets",
    "tabulate",
    "jupyterlab-spreadsheet-editor"
]

requirements = "\n".join(requirements)
with open("./requirements.txt", "w") as f:
    f.write(requirements)

In [27]:
%pip install -r requirements.txt

  Using cached tabulate-0.10.0-py3-none-any.whl.metadata (40 kB)
Using cached tabulate-0.10.0-py3-none-any.whl (39 kB)
Note: you may need to restart the kernel to use updated packages.


In [10]:
import subprocess
import sys
import os
import pandas as pd
from Bio import Entrez, Medline
import ssl
import certifi
import re
from pathlib import Path

In [29]:
root = os.getcwd()
folders = {
"systematic_review": f"{root}/systematic_review",
    "protocol": f"{root}/systematic_review/protocol",
        "prospero": f"{root}/systematic_review/protocol/prospero",
        "cochrane": f"{root}/systematic_review/protocol/cochrane",
    "search_strategy": f"{root}/systematic_review/search_strategy",
        "search_strategy_pubmed": f"{root}/systematic_review/search_strategy/pubmed/",
        "search_strategy_embase": f"{root}/systematic_review/search_strategy/embase/",
        "search_strategy_wos": f"{root}/systematic_review/search_strategy/wos/",
    "search": f"{root}/systematic_review/search",
        "search_pubmed": f"{root}/systematic_review/search/pubmed/",
        "search_embase": f"{root}/systematic_review/search/embase/",
        "search_wos": f"{root}/systematic_review/search/wos/",
    "deduplication": f"{root}/systematic_review/deduplication/",
    "screening": f"{root}/systematic_review/screening/",
        "title_abstract_screening": f"{root}/systematic_review/screening/title_abstract_screening", 
        "pdf": f"{root}/systematic_review/screening/PDF",
        "full_text_screening": f"{root}/systematic_review/screening/full_text_screening", 
"meta-analysis": f"{root}/meta-analysis",
"manuscript": f"{root}/manuscript"    
}

for var, f in folders.items():
    relative = "./"
    absolute = os.getcwd()
    directory = Path(f)
    globals()[f"{var}"] = directory
    path = Path(relative, directory)
    os.makedirs(path, exist_ok = True)

In [61]:
dfs = [
    "pm_bptb",
    "pm_ht",
    "pm_qt",
    "pm_plt",
    "pm_at",
    "pm_ta",
    "em_bptb",
    "em_ht",
    "em_qt",
    "em_plt",
    "em_at",
    "em_ta",
    "wos_bptb",
    "wos_ht",
    "wos_qt",
    "wos_plt",
    "wos_at",
    "wos_ta",
]

url = "https://raw.githubusercontent.com/d-wkim/d-wkim/refs/heads/main/data/"

for filename in dfs:
    data = pd.read_csv(f"{url}/{filename}.csv", encoding = "utf-8")
    globals()[f"{filename}"] = data

pm_bptb.head()

bptb = pd.concat([pm_bptb, em_bptb, wos_bptb])
ht = pd.concat([pm_ht, em_ht, wos_ht])
 = pd.concat([pm_, em_, wos_])
at = pd.concat([pm_at, em_at, wos_at])
plt = pd.concat([pm_, pltem_, wos_]plt)
qt = pd.concat([pm_qt, em_, wos_qt])


[top](#toc) | [next](#search)  
[search strategy](#search-strategy) | [search](#search) | [deduplication](#deduplication) | [screening](#screening)


<a id="systematic_review"></a>

<div>
    <h1 align="center" style="font-family:Times New Roman;font-variant:small-caps;">
                Systematic Review
    </h1>
</div>

<a id="search-strategy"></a>

<div align="center">
    <h2 align="center" style="font-family:Times New Roman;font-variant: small-caps;">
        Search Strategy
    </h2>
    </br></div>

<p>
    Search strategies were developed for randomized controlled trials: `pm_bptb.txt`, `pm_ht.txt`, `pm_qt.txt`, `pm_plt.txt`, `pm_at.txt` and `pm_ta.txt` corresponding to PubMed search strategies for patellar, hamstring, quadriceps, peroneus longus, achilles and tibialis anterior and posterior tendones, respectively. From the protocol that was developed, extract key terms from the eligibility criteria for inclusion and exclusion of studies in order to develop a *search strategy*.
</p>

<p>
    The search strategies were 'translated' via regular expressions from PubMed syntax to Embase and Web of Science syntax, store them into the global environment, and save them as plain text files for importing and use as queries for search.
</p>

**Search strategies**

|       |              |                                                                                           |
| :---: | :----------- | :---------------------------------------------------------------------------------------- |
| **P** | Population   | ```NOT (("pediatric"[tiab] OR "paediatric"[tiab]) OR ("revision"[tiab] OR "repair"[tiab]))``` |
| **I** | Intervention | ```("anterior cruciate ligament"[mh] OR "anterior cruciate ligament"[tiab] OR "anterior cruciate ligament reconstruction"[tiab] OR "acl"[tiab])``` |
| **C** | Comparators  | ```("bone-patellar tendon-bone"[tiab] OR "patellar tendon"[tiab] OR "bptb"[tiab])``` <br> ```("hamstring tendon"[tiab] OR "semitendinosus"[tiab] OR "gracilis"[tiab])``` <br> `("quadriceps"[tiab] OR "quadriceps tendon"[tiab] OR "qt"[tiab])` <br> `("peroneus longus"[tiab] OR "fibularis longus"[tiab])` <br> `("achilles"[tiab])` <br> `("tibialis anterior"[tiab] OR "tibialis posterior"[tiab]` |
| **O** | Outcomes     | `("ikdc"[tiab] OR "lysholm"[tiab] OR "tegner"[tiab] OR (("instrumental laxity"[tiab] OR "kt-1000"[tiab] OR "kt-2000"[tiab] OR "rolimeter"[tiab]) OR "pivot shift"[tiab] OR "lachman"[tiab]) OR ("graft failure"[tiab] OR "graft rupture"[tiab]))`                   |
| **S** | Study design | `("randomized controlled trial"[pt] OR "randomized controlled trial"[tiab] OR "randomised controlled trial"[tiab]) NOT ("review"[pt] OR "review"[tiab] OR "systematic review"[pt] OR "systematic review"[tiab] OR "meta-analysis"[pt] OR "meta-analysis"[tiab])`   |

In [1]:
population = f"""(("pediatric"[tiab] OR "paediatric"[tiab]) OR ("revision"[tiab] OR "repair"[tiab]))"""
acl = f"""("anterior cruciate ligament"[mh] OR "anterior cruciate ligament"[tiab] OR "anterior cruciate ligament reconstruction"[tiab] OR "acl"[tiab])"""
rct = f"""("randomized controlled trial"[pt] OR "randomized controlled trial"[tiab] OR "randomised controlled trial"[tiab])"""
reviews = f"""("review"[pt] OR "review"[tiab] OR "systematic review"[pt] OR "systematic review"[tiab] OR "meta-analysis"[pt] OR "meta-analysis"[tiab])"""
outcomes = f"""("ikdc"[tiab] OR "lysholm"[tiab] OR "tegner"[tiab] OR (("instrumental laxity"[tiab] OR "kt-1000"[tiab] OR "kt-2000"[tiab] OR "rolimeter"[tiab]) OR "pivot shift"[tiab] OR "lachman"[tiab]) OR ("graft failure"[tiab] OR "graft rupture"[tiab]))"""

bptb = f"""("bone-patellar tendon-bone"[tiab] OR "bone patellar tendon bone"[tiab] OR "patellar tendon"[tiab] OR "bptb"[tiab])"""
ht = f"""("hamstring"[tiab] OR "hamstring tendon"[tiab] OR "semitendinosus"[tiab] OR "gracilis"[tiab])"""
qt = f"""("quadriceps"[tiab] OR "quadriceps tendon"[tiab] OR "qt"[tiab])"""
plt = f"""("peroneus"[tiab] OR "peroneus longus"[tiab] OR "fibularis longus"[tiab])"""
at = f"""("achilles"[tiab] OR "achilles tendon"[tiab])"""
ta = f"""("tibialis"[tiab] OR "tibialis anterior"[tiab] OR "tibialis posterior"[tiab])"""

In [2]:
subgroups = {"bptb": bptb, 
             "ht": ht, 
             "qt": qt, 
             "plt": plt, 
             "at": at, 
             "ta": ta}

queries = {}

for x, y in subgroups.items():
    query = f"{acl} AND {y} AND ({rct} NOT {reviews}) AND {outcomes}"
    globals()[f"pm_{x}"] = query
    with open(f"{search_strategy}/pubmed/pm_{x}.txt", 'w') as f:
        f.write(query)

pm_queries = {
    "bptb": pm_bptb,
    "ht": pm_ht,
    "qt": pm_qt,
    "plt": pm_plt,
    "at": pm_at,
    "ta": pm_ta
}

for x, y in pm_queries.items():
    query = re.sub(r"\[(.*?)\]",":\\1", y)
    query = re.sub(r"\:tiab",":ti,ab", query)
    query = re.sub(r"\:mh","/exp", query)
    query = re.sub(r"\:pt",":it,ti,ab", query)
    query = re.sub(r'"',"'", query)
    globals()[f"em_{x}"] = query    
    with open(f"{search_strategy}/embase/em_{x}.txt", 'w') as f:
        f.write(query)


for x, y in pm_queries.items():
    query = re.sub(r'"(.*?)"\[(.*?)\]','\\2="\\1"', y)
    query = re.sub(r'tiab="(.*?)"','(TI=(\\1) OR AB=(\\1))', query)
    query = re.sub(r'mh="(.*?)"','(TMIC=(\\1))', query)
    query = re.sub(r'pt="(.*?)"','(TS=(\\1))', query)
    globals()[f"wos_{x}"] = query
    with open(f"{search_strategy}/wos/wos_{x}.txt", 'w') as f:
        f.write(query)

NameError: name 'search_strategy' is not defined

In [25]:
queries = [
    pm_bptb, pm_ht, pm_qt, pm_plt, pm_ta, pm_at,
    em_bptb, em_ht, em_qt, em_plt, em_ta, em_at,
    wos_bptb, wos_ht, wos_qt, wos_plt, wos_ta, wos_at
]

for query in queries:
    print("\n\n", query)

df = pd.DataFrame({
    "Categories": ["Population", "Intervention", "Comparators", "Outcomes", "Study Design"]
})

df.to_markdown()



 ("anterior cruciate ligament"[mh] OR "anterior cruciate ligament"[tiab] OR "anterior cruciate ligament reconstruction"[tiab] OR "acl"[tiab]) AND ("bone-patellar tendon-bone"[tiab] OR "bone patellar tendon bone"[tiab] OR "patellar tendon"[tiab] OR "bptb"[tiab]) AND (("randomized controlled trial"[pt] OR "randomized controlled trial"[tiab] OR "randomised controlled trial"[tiab]) NOT ("review"[pt] OR "review"[tiab] OR "systematic review"[pt] OR "systematic review"[tiab] OR "meta-analysis"[pt] OR "meta-analysis"[tiab])) AND ("ikdc"[tiab] OR "lysholm"[tiab] OR "tegner"[tiab] OR (("instrumental laxity"[tiab] OR "kt-1000"[tiab] OR "kt-2000"[tiab] OR "rolimeter"[tiab]) OR "pivot shift"[tiab] OR "lachman"[tiab]) OR ("graft failure"[tiab] OR "graft rupture"[tiab]))


 ("anterior cruciate ligament"[mh] OR "anterior cruciate ligament"[tiab] OR "anterior cruciate ligament reconstruction"[tiab] OR "acl"[tiab]) AND ("hamstring"[tiab] OR "hamstring tendon"[tiab] OR "semitendinosus"[tiab] OR "gracil

ImportError: `Import tabulate` failed.  Use pip or conda to install the tabulate package.

[previous](#search-strategy) | [top](#toc) | [next](#deduplication)  
[search strategy](#search-strategy) | [search](#search) | [deduplication](#deduplication) | [screening](#screening)

<!--
<div align="center"><font size="5" style="font-family:Times New Roman;font-variant: small-caps;">Search</font><br></div>
-->

<a id="search"></a>

<h2 align="center" style="font-family:Times New Roman;font-variant:small-caps;">Search</h2>

---

A script to either create a search strategy using the terms, field tags, and Boolean operators and save them as plain text files or load already written and saved plain text files for import into the API search scripts. This uses the search strategies (e.g., `pm_bptb.txt`, search strategy in plain text written in PubMed syntax for bone-patellar tendon-bone (BPTB) subgroup search) and pulls data from PubMed to output PMIDs (`pmid_pm_bptb.txt`) and search results in XML (`pm_bptb.xml`) and parses this into CSV files (`pm_bptb.csv`).

In [11]:
ssl._create_default_https_context = lambda: ssl.create_default_context(
    cafile=certifi.where()
)

# create search strategy using structured inputs

question = input("Do you already have a search strategy file saved?") # get rid of this
filename = input("Enter the file name of the search strategy: ") # use file upload widget ! and present the results as data table widgets !!! 
file = f"{search_strategy}/pubmed/{filename}.txt"

if question == "no":
    parts = []
    string = []
    while True:
        term = input("Enter the search string: ")
        field = input("Enter the field type: ")
        string.append(f"'{term}'[{field}]")
        boolean = input("Enter the Boolean operator (e.g., OR, AND, NOT): ")
        
        if boolean == "OR":
            continue
    
        parts.append("(" + " OR ".join(string) + ")")
        string = []
    
        if boolean == "":
            break
            
        parts.append(boolean)
        
    query = " ".join(parts) # query = search strategy FROM HERE
    with open(file, "w") as f:
        f.write(query)
    
with open(file, "r") as f:
    query = f.read()

query = f"{query}"

# use NCBI's e-utitilies to pull PMIDs using e-search.

Entrez.email = "dkim246@jhmi.edu"
Entrez.api_key = 'bb1c481d8e167acd16f3616593c18b3aab08'

handle = Entrez.esearch(db= "pubmed", 
                        term = query, 
                        usehistory = "y", 
                        retmax = 2000,
                        retmode = "xml")

pmid = Entrez.read(handle)

pmid = pmid['IdList']
pmid = ",".join(pmid) # list to string
#with open(f"./data/pmid_{filename}.txt", 'w') as f:
#    f.write(pmid)
os.makedirs(f"{search}/pubmed/pmid/", exist_ok = True)
with open(f"{search}/pubmed/pmid/{filename}.txt", 'w') as f:
    f.write(pmid)
handle.close()

# ncbi e-summary
handle = Entrez.esummary(db= "pubmed", 
                         id = pmid, 
                         retmode = "xml", 
                         usehistory = "y", 
                         retmax = 2000)

xml = handle.read()
#xml_file = f"./data/{filename}.xml"
os.makedirs(f"{search}/pubmed/xml/", exist_ok = True)
xml_file = f"{search}/pubmed/xml/{filename}.xml"
with open(xml_file, "wb") as f:
    f.write(xml)   
handle.close()

import xml.etree.ElementTree as ET

tree = ET.parse(f"{xml_file}")
root = tree.getroot()

docsum = root[0]

def xml_parse(docsum):
    df = {}
    df["pmid"] = docsum.find("Id").text
    for item in docsum.findall("Item"):
        key = item.attrib.get("Name")
        if item.attrib.get("Type") == "List":
            values = [sub.text for sub in item.findall("Item") if sub.text]
            df[key] = values
        else:
            df[key] = item.text
    return df
records = [xml_parse(doc) for doc in root.findall(".//DocSum")]
df = pd.DataFrame(records)

os.makedirs(f"{search}/pubmed/", exist_ok = True)
csv_file = f"{search}/pubmed/{filename}.csv"
df.to_csv(csv_file, encoding = "utf-8")

# using e-fetch, the abstracts are pulled

handle = Entrez.efetch(
    db="pubmed",
    id=pmid,
    rettype="medline",
    retmode="text"
)

text = list(Medline.parse(handle))
data = pd.DataFrame(text)
data_csv = data.map(lambda x: ", ".join(map(str, x)) if isinstance(x, list) else x)
os.makedirs(f"{search}/pubmed/medline/", exist_ok = True)
data_csv.to_csv(f"{search}/pubmed/medline/{filename.replace("pm","md")}.csv", index=False)
globals()[f"{filename.replace("pm","md")}"] = data
handle.close()

abstracts = pd.DataFrame(text)[["PMID", "AB"]]
abstracts.rename(columns = {"PMID":"pmid", "AB":"abstract"}, inplace = True)
df = df.merge(abstracts, on = "pmid", how = "left")
df['year'] = df['PubDate'].str[:4]
csv_file = f"{search}/pubmed/{filename}.csv"
df.to_csv(csv_file, encoding = "utf-8")
num = len(df)
print(f"Number of records found: {num}")
data.info()

Do you already have a search strategy file saved? yes
Enter the file name of the search strategy:  pm_bptb


Number of records found: 233
<class 'pandas.DataFrame'>
RangeIndex: 233 entries, 0 to 232
Data columns (total 52 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   PMID    233 non-null    str   
 1   OWN     233 non-null    str   
 2   STAT    233 non-null    str   
 3   LR      233 non-null    str   
 4   IS      233 non-null    str   
 5   DP      233 non-null    str   
 6   TI      233 non-null    str   
 7   LID     117 non-null    str   
 8   AB      231 non-null    str   
 9   CI      47 non-null     object
 10  FAU     233 non-null    object
 11  AU      233 non-null    object
 12  AUID    14 non-null     object
 13  AD      230 non-null    object
 14  LA      233 non-null    object
 15  GR      11 non-null     object
 16  PT      233 non-null    object
 17  DEP     118 non-null    str   
 18  PL      233 non-null    str   
 19  TA      233 non-null    str   
 20  JT      233 non-null    str   
 21  JID     233 non-null    str   
 22  SB      

The CSV files from the three databases were now cleaned, prepared, and transformed; this is otherwise known as 'data wrangling' in Data Science.

Step 1: import the 18 datasets for each subgroup from database search.  
Step 2: rename the columns for each dataset and merge them into 3 separate datasets, one for each database.  
Step 3: merge the datasets, with matching columns into 1 dataset.  
Step 4: clean and prepare the records dataset (as it is the most important output).  
Step 5: save records as csv files into the appropriate directory.   



```mermaid
---
config:
  theme: light
  curve: step
---

flowchart LR

A01["`**RCT**<br>rct_pm_bptb`"]
B01["`**RCT**<br>rct_pm_ht`"]
C01["`**RCT**<br>rct_pm_qt`"]
D01["`**RCT**<br>rct_pm_plt`"]
E01["`**RCT**<br>rct_pm_at`"]
F01["`**RCT**<br>rct_pm_ta`"]

A02["reviews_pm_bptb"]
B02["reviews_pm_ht"]
C02["reviews_pm_qt"]
D02["reviews_pm_plt"]
E02["reviews_pm_at"]
F02["reviews_pm_ta"]

A1["pm_bptb"]
B1["pm_ht"]
C1["pm_qt"]
D1["pm_plt"]
E1["pm_at"]
F1["pm_ta"]

A2["em_bptb"]
B2["em_ht"]
C2["em_qt"]
D2["em_plt"]
E2["em_at"]
F2["em_ta"]

A3["wos_bptb"]
B3["wos_ht"]
C3["wos_qt"]
D3["wos_plt"]
E3["wos_at"]
F3["wos_ta"]

G["pubmed"]
H["embase"]
I["wos"]

 
A02 & A01 --> A1
B02 & B01 --> B1
C02 & C01 --> C1
D02 & D01 --> D1
E02 & E01 --> E1
F02 & F01 --> F1

A1 & B1 & C1 & D1 & E1 & F1 --> G
A2 & B2 & C2 & D2 & E2 & F2 --> H
A3 & B3 & C3 & D3 & E3 & F3 --> I

G & H & I --> J["records"]
```

Step 1: import the 18 datasets for each subgroup from database search and load them into the global environment.

In [11]:

subgroups = {
    "bptb": "patellar",
    "ht": "hamstring",
    "qt": "quadriceps",
    "plt": "peroneus",
    "at": "achilles",
    "ta": "tibialis"}

for x, y in subgroups.items():
    df = pd.read_csv(f"{search}/wos/tsv/wos_{x}.tsv", sep = '\t', encoding = "latin-1")
    df.to_csv(f"{search}/wos/wos_{x}.csv", encoding = "utf-8", sep = ",", index = False)
    globals()[f"wos_{x}"] = df

databases = {"pm": "pubmed", 
             "em": "embase", 
             "wos": "wos"}

subgroups = {"bptb": "patellar", 
             "ht": "hamstring", 
             "qt": "quadriceps", 
             "plt": "peroneus", 
             "at": "achilles", 
             "ta": "tibialis"}

for a, b in databases.items():
    dfs = []
    for x, y in subgroups.items():
        df = pd.read_csv(f"{search}/{b}/{a}_{x}.csv", encoding = "utf-8")
        df['source'] = b
        df['subgroup'] = x
        dfs.append(df)
        globals()[f"{a}_{x}"] = df
        data = pd.concat(dfs, ignore_index = True)
        data.insert(0, "id", range(1, len(data) + 1))
        data.to_csv(f"{search}/{b}/{b}.csv", encoding = "utf-8", index = False)
        globals()[f"{b}"] = data # saving as variables

Step 2: rename the columns for each dataset and merge them into 3 separate datasets, one for each database.  

In [12]:
databases = {"pm": "pubmed", 
             "em": "embase", 
             "wos": "wos"}

subgroups = {"bptb": "patellar", 
             "ht": "hamstring", 
             "qt": "quadriceps", 
             "plt": "peroneus", 
             "at": "achilles", 
             "ta": "tibialis"}

embase.rename(columns = {
	"Title" : "title",
	"Original Title" : "original_title",
	"Author Names" : "authors",
	"Author Addresses" : "author_addresses",
	"Correspondence Address" : "correspondence_address",
	"Editors" : "editors",
	"AiP/IP Entry Date" : "aip/ip_entry_date",
	"Full Record Entry Date" : "full_record_entry_date",
	"Source" : "journal_full",
	"Source title" : "journal",
	"Publication Year" : "year",
	"Volume" : "volume",
	"Issue" : "issue",
	"First Page" : "first_page",
	"Last Page" : "last_page",
	"Date of Publication" : "date",
	"Publication Type" : "study_design",
	"Conference Name" : "conference_name",
	"Conference Location" : "conference_location",
	"Conference Date" : "conference_date",
	"Conference Editors" : "conference_editors",
	"ISSN" : "issn",
	"ISBN" : "isbn",
	"Name" : "name",
	"Location" : "location",
	"Date" : "date",
	"Editors" : "editors",
	"Book Publisher" : "book_publisher",
	"Abstract" : "abstract",
	"Original Abstract" : "original_abstract",
	"Author Keywords" : "author_keywords",
	"Emtree Drug Index Terms (Major Focus)" : "emtree_drug_index_terms_(major_focus)",
	"Emtree Drug Index Terms" : "emtree_drug_index_terms",
	"Emtree Medical Index Terms (Major Focus)" : "emtree_medical_index_terms_(major_focus)",
	"Emtree Medical Index Terms" : "emtree_medical_index_terms",
	"Drug Tradenames" : "drug_tradenames",
	"Drug Manufacturer" : "drug_manufacturer",
	"Device Tradenames" : "device_tradenames",
	"Device Manufacturer" : "device_manufacturer",
	"CAS Registry Numbers" : "cas_registry_numbers",
	"Molecular Sequence Numbers" : "molecular_sequence_numbers",
	"Embase Classification" : "embase_classification",
	"Clinical Trial Numbers" : "clinical_trial_numbers",
	"Article Language" : "language",
	"Summary Language" : "summary_language",
	"Embase Accession ID" : "embase_accession_id",
	"Medline PMID" : "pmid",
	"PUI" : "pui",
	"DOI" : "doi",
	"Full Text Link" : "full_text_link",
	"Embase Link" : "embase_link",
	"Open URL Link" : "open_url_link",
	"Copyright" : "copyright",
	"source" : "source",
	"subgroup" : "subgroup"
}, inplace = True)

pubmed.rename(columns = {
	"id" : "id",
	"pmid" : "pmid",
	"PubDate" : "date",
	"EPubDate" : "epubdate",
	"Source" : "journal_abbr",
	"AuthorList" : "authors",
	"LastAuthor" : "last_author",
	"Title" : "title",
	"Volume" : "volume",
	"Issue" : "issue",
	"Pages" : "pages",
	"LangList" : "language",
	"NlmUniqueID" : "nlmuniqueid",
	"ISSN" : "issn",
	"ESSN" : "essn",
	"PubTypeList" : "study_design",
	"RecordStatus" : "recordstatus",
	"PubStatus" : "pubstatus",
	"Articlelds" : "articlelds",
	"DOI" : "doi",
	"History" : "history",
	"References" : "references",
	"HasAbstract" : "hasabstract",
	"PmcRefCount" : "pmcrefcount",
	"FullJournalName" : "journal",
	"ELocationID" : "elocationid",
	"SO" : "so",
	"abstract" : "abstract",
	"source" : "source",
	"subgroup" : "subgroup"
}, inplace = True)

wos.rename(columns = {
    "id": "id",
	"ï»¿PT" : "study_design",
	"AU" : "authors",
	"BA" : "book_authors",
	"BE" : "book_editors",
	"GP" : "book_group_authors",
	"AF" : "authors_full",
	"BF" : "book_author_full_names",
	"CA" : "group_authors",
	"TI" : "title",
	"SO" : "journal",
	"SE" : "book_series_title",
	"BS" : "book_series_subtitle",
	"LA" : "language",
	"DT" : "document_type",
	"CT" : "conference_title",
	"CY" : "conference_date",
	"CL" : "conference_location",
	"SP" : "conference_sponsor",
	"HO" : "conference_host",
	"DE" : "author_keywords",
	"ID" : "keywords_plus",
	"AB" : "abstract",
	"C1" : "addresses",
	"C3" : "affiliations",
	"RP" : "reprint_addresses",
	"EM" : "email_addresses",
	"RI" : "researcher_ids",
	"OI" : "orcids",
	"FU" : "funding_orgs",
	"FP" : "funding_name_preferred",
	"FX" : "funding_text",
	"CR" : "cited_references",
	"NR" : "cited_reference_count",
	"TC" : "times_cited, wos_core",
	"Z9" : "times_cited, all_databases",
	"U1" : "180_day_usage_count",
	"U2" : "since_2013_usage_count",
	"PU" : "publisher",
	"PI" : "publisher_city",
	"PA" : "publisher_address",
	"SN" : "issn",
	"EI" : "eissn",
	"BN" : "isbn",
	"J9" : "journal_9",
	"JI" : "journal_abbr",
	"PD" : "date",
	"PY" : "year",
	"VL" : "volume",
	"IS" : "issue",
	"PN" : "part_number",
	"SU" : "supplement",
	"SI" : "special_issue",
	"MA" : "meeting_abstract",
	"BP" : "start_page",
	"EP" : "end_page",
	"AR" : "article_number",
	"DI" : "doi",
	"DL" : "doi_link",
	"D2" : "book_doi",
	"EA" : "early_access_date",
	"PG" : "number_of_pages",
	"WC" : "wos_categories",
	"WE" : "web_of_science_index",
	"SC" : "research_areas",
	"GA" : "ids_number",
	"PM" : "pmid",
	"OA" : "open_access_designations",
	"HC" : "highly_cited_status",
	"HP" : "hot_paper_status",
	"DA" : "date_of_export",
	"UT" : "ut (unique_wos_id)",
	"source" : "source",
    "subgroup": "subgroup"
}, inplace = True)

# Step 3: write the CSV files
pubmed.to_csv(f"{search}/pubmed.csv", encoding = "utf-8")
embase.to_csv(f"{search}/embase.csv", encoding = "utf-8")
wos.to_csv(f"{search}/wos.csv", encoding = "latin-1")

In [18]:
a = len(pubmed)
b = len(embase)
c = len(wos)

a + b + c

1563

## Step 3: clean and prepare the datasets before merging them into 1 dataset.  

In [ ]:
databases = {
    'pubmed': pubmed, 
    'embase': embase,
    'wos': wos
}

for text, var in databases.items():
    df = pd.DataFrame({
        "id": var["id"],
        "pmid": var["pmid"],
        "source": var["source"],
        "subgroup": var["subgroup"],
        "doi": var["doi"],
        "authors": var["authors"].str.replace(r"[\['\]]","",regex=True),
        "journal": var["journal"],
        "title": var["title"],
        "abstract": var["abstract"],
        "year": var["year"],
        "language": var["language"]
    })
    df['pmid'] = df['pmid'].astype(str)
    df['language'] = df['language'].str.replace(r"[\'\[\]]","", regex = True)
    df.fillna("")
    globals()[f"{text}"] = df


# authors
pubmed["first_author"] = pubmed["authors"].str.replace(r"[\'\[\].;]","", regex = True).str.split(r",\s*").str[0]
pubmed["second_author"] = pubmed["authors"].str.replace(r"[\'\[\].;]","", regex = True).str.split(r",\s*").str[1]

embase["authors"] = embase["authors"].str.replace(r"\.", "", regex = True)
embase["first_author"] = embase["authors"].str.replace(r"[\'\[\].;]","", regex = True).str.split(r",\s*").str[0]
embase["second_author"] = embase["authors"].str.replace(r"[\'\[\].;]","", regex = True).str.split(r",\s*").str[1]

wos["authors"] = wos["authors"].str.replace(r",","", regex = True)
wos["authors"] = wos["authors"].str.replace(r";",",", regex = True)
wos["first_author"] = wos["authors"].str.replace(r"[\'\[\].;]","", regex = True).str.split(r",\s*").str[0]
wos["second_author"] = wos["authors"].str.replace(r"[\'\[\].;]","", regex = True).str.split(r",\s*").str[1]

pubmed.head()

In [ ]:
pubmed['authors'] = pubmed['authors'].str.replace(r"[\'\[\]]","", regex = True)
pubmed['journal'] = pubmed['journal'].str.replace(r"[\'\[,\]]","", regex = True)
pubmed['journal'] = pubmed['journal'].str.replace(r"\(.*?\)","", regex=True)
pubmed['journal'] = pubmed['journal'].str.capitalize()

embase['doi'] = embase['doi'].fillna("")
embase['pmid'] = embase['pmid'].fillna("")
pubmed['journal'] = pubmed['journal'].str.replace(r"\(.*?\)","", regex=True)
embase['journal'] = embase['journal'].str.replace(r"[\'\[,\]]","", regex = True)
embase['journal'] = embase['journal'].str.capitalize()

wos['journal'] = wos['journal'].str.replace(r"[\'\[\]]","", regex = True)
wos['journal'] = wos['journal'].str.capitalize()

records = pd.concat([pubmed, embase, wos], ignore_index = True)

Step 5: Clean and prepare the records dataset for the next stage.

In [ ]:
wos.head()

In [ ]:
records['short_title'] = records['title'].str.replace(r'[\[\]\s,.;-]','',regex = True).str.lower().str[:60]
records['title+author+year'] = records['first_author'] + '+' + records['short_title'] + '+' + records['year'].astype(str)
records['title+year'] = records['short_title'] + '+' + records['year'].astype(str)
records['language'] = records['language'].str.replace(r"[\'\[\]]","", regex = True)
records["subgroup"] = records["subgroup"].str.upper()
records["doi_url"] = f"https://doi.org/" + records["doi"]
records["pmid_url"] = "https://pubmed.ncbi.nlm.nih.gov/" + records["pmid"].astype(str) + "/"
records["study"] = records['first_author'] + " (" + records['year'].astype(str) + ")"
records = records[["id", "study", "subgroup", "authors", "first_author", "title", "short_title", "abstract", "year", "language", "journal", "source", "doi", "doi_url", "pmid", "pmid_url", "title+author+year", "title+year"]]

records['year'] = records['year'].astype(str)
records['pmid'] = round(records['pmid'],0)
records['pmid'] = records['pmid'].astype(str)
records['first_author'] = records['first_author'].astype(str)
records['year'] = records['year'].astype(str)
records.rename(columns = {"id":"source_id"}, inplace = True)
records["id"] = range(1,len(records)+1)
records = records[["id", "source_id", "study", "subgroup", "authors", "first_author", "title", "short_title", "abstract", "year", "language", "journal", "source", "doi", "doi_url", "pmid", "pmid_url",  "title+author+year", "title+year"]]

records['first_author'] = authors_split.str[0].str.strip().str.split().str[0]
records['second_author'] = authors_split.str[1].fillna('').str.strip().str.split().str[0]
records['num_authors'] = authors_split.str.len()

records.loc[records['num_authors'] >= 3, 'study'] = (
    records['first_author'] + ' et al. (' + records['year'].astype(str) + ')'
)

records = records.sort_values(by = ['subgroup', 'year', 'authors'])
records["authors"] = records["authors"].fillna("")
records = records.fillna("")
records = records.sort_values(by = ['authors'])
records = records.sort_values(by = ['subgroup'])
records = records.sort_values(by = ['year'], ascending = False)
records.to_csv(f"{search}/records.csv", encoding= "utf-8")
records.to_csv(f"{deduplication}/records.csv", encoding = "utf-8")

[previous](#search) | [top](#toc) | [next](#screening)  
[search strategy](#search-strategy) | [search](#search) | [deduplication](#deduplication) | [screening](#screening)

<a id="deduplication"></a>
<h2 align="center" style="font-family:Times New Roman;font-variant: small-caps;">Deduplication</h2>

The 'gold-standard' or consensus agreement among researchers seems to converge on the idea that removal of duplicate records is best performed in a process that involves three ordered stages. The first is deduplication based on a unique record identifier, such as a digital object identifier (DOI) number, PubMed identifier (PMID) number, or clinicaltrials.gov (NCT) number. Then, as according to as described for the second stage, the remaining records were deduplicated based on a concatenated column consisting of the title, author, and year. The title was standardized by converting to sentence case, and punctuation marks and white spaces were removed. The character length was decreased to 50. For the authors column, the last name of the first author was chosen to be used. For the year of publication, the year was extracted from the date of publication in the electronic version of the journal and converted into a string data structure. 

input file(s): `records.csv`, output file(s): `doi_deduplicated.csv`,
`pmid_deduplicated.csv`, `title+author+year_deduplicated.csv`, and `title+year_deduplicated.csv`.

In [1]:
import pandas as pd
import mermaid
import os

input_file_name = f"{root}/systematic_review/deduplication/records.csv"

#input_file_name = f"{deduplication}/" + input('Enter file name: ') + '.csv'

deduplication = f"{root}/systematic_review/deduplication",

df = pd.read_csv(input_file_name) # A (records)
cols_input = input('Enter the column for which to deduplicate based on: ')
cols = [c.strip() for c in cols_input.split(',')]
folder = '_'.join(cols)
os.makedirs(f"{deduplication}/{folder}/", exist_ok = True)

output_file = f"{deduplication}/{folder}/{folder}_deduplicated"
output_file_name = f"{output_file}.csv"
output_for_recycle = f"{deduplication}/{folder}_deduplicated.csv"
prisma_file_name = f"{output_file}.mmd"

nulls_mask = df[cols].isnull().any(axis=1)
df_nulls = df[nulls_mask] # B
df_non_nulls = df[~nulls_mask] # C

duplicates_mask = df_non_nulls.duplicated(subset = cols, keep = False)
df_non_duplicates = df_non_nulls[~duplicates_mask] # D
df_duplicates = df_non_nulls[duplicates_mask] # E
#df_duplicates.groupby(cols, as_index=False).agg(agg_map)
df_kept = df_duplicates.drop_duplicates(subset = cols, keep = 'first')
#df_kept = df_duplicates.groupby(cols, as_index=False).agg(lambda s: list(dict.fromkeys(s.dropna())) if s.name in ['subgroup', 'source'] else s.dropna().iloc[0] if len(s.dropna()) else pd.NA)
#df_kept = df_duplicates.groupby(cols, as_index=False).agg(lambda s: list(dict.fromkeys(s.dropna())) if s.name == 'subgroup' else s.dropna().iloc[0] if len(s.dropna()) else pd.NA)
df_removed = df_duplicates[~df_duplicates.index.isin(df_kept.index)]
#df_kept = df_duplicates.groupby(cols, as_index=False).agg(lambda s: '; '.join(dict.fromkeys(s.dropna().astype(str).str.strip())) if s.name == 'subgroup' and s.dropna().astype(str).str.strip().nunique() > 1 else (s.dropna().astype(str).str.strip().iloc[0] if len(s.dropna().astype(str).str.strip()) else pd.NA))
df_unique = df_non_nulls.drop_duplicates(subset = cols, keep = 'first') # df of unique
df_deduplicated = pd.concat([df_non_duplicates, df_kept, df_nulls], ignore_index=True) # df of unique + df of non-duplicates

results = {"records": len(df),  
"nulls": len(df_nulls), 
"non_nulls": len(df_non_nulls), 
"non_duplicates": len(df_non_duplicates), 
"duplicates": len(df_duplicates), 
"removed": len(df_removed), 
"kept": len(df_kept),
"unique": len(df_unique),
"deduplicated": len(df_deduplicated)
}

output_file_name = f"{deduplication}/deduplicated.csv"
df_nulls.to_csv(output_file_name.replace('deduplicated','nulls'), index = False)
df_deduplicated.to_csv(output_file_name, index = False)
df_removed.to_csv(output_file_name.replace('deduplicated','duplicates_removed'), index = False)
df_deduplicated.to_csv(output_for_recycle, index = False)

graph_text = f"""---
config:
theme: neutral
curve: stepBefore
---
graph TD;
A["`**records** (*n* = {results['records']})`"];
B["`null (*n* = {results['nulls']})`"];
C["`non-null (*n* = {results['non_nulls']})`"];
D["`non-duplicates (*n* = {results['non_duplicates']})`"];
E["`duplicates (*n* = {results['duplicates']})`"];
F["`duplicates kept (*n* = {results['kept']})`"];
G["`duplicates removed (*n* = {results['removed']})`"];
H["`unique (*n* = {results['unique']})`"];
I["`deduplicated (*n* = {results['deduplicated']})`"];

A --> B & C;
C --> D & E;
E --> F & G;
D & F --> H
B & H --> I"""

with open(prisma_file_name, "w") as f:
    f.write(graph_text)

!mmdc -i "{prisma_file_name}" -o "{output_file}"_light.svg
!mmdc -i "{prisma_file_name}" -o "{output_file}"_dark.svg -t dark -b transparent
print(results)

NameError: name 'root' is not defined

[top](#toc) | [search strategy](#search-strategy) | [search](#search) | [deduplication](#deduplication) | [screening](#screening) |
[data collection](#data-collection) |

<a id="screening"></a>

<h2 align="center" style="font-family:Times New Roman;font-variant: small-caps;">Screening</h2>

---

<h3 align="center" style="font-family:Times New Roman;font-variant:small-caps;">Title abstract screening</h3>

In [12]:
%pip install -q ipywidgets
import ipywidgets as widgets
import pandas as pd

df = csv2df('records')

df = pd.read_csv(f"./records.csv", encoding = "utf-8")
columns = df.columns
", ".join(columns)

text = f"""id, source_id, study, subgroup, authors, first_author, title, short_title, abstract, year, language, journal, source, doi, doi_url, pmid, pmid_url, title+author+year, title+year, second_author, num_authors'"""
list = text.split(", ")
list

A = widgets.IntText(value=0, description="Study ID ", layout={"width":"20%"})
B = widgets.HTML(value="", layout={"width":"80%", "height":"100%"})
C = widgets.HTML(value="", layout={"width":"80%", "height":"100%"})

def update(change):
    title = df.loc[df["id"] == A.value, "title"]
    abstract = df.loc[df["id"] == A.value, "abstract"]

    if  A.value > 0:
        B.value = f"<p>{str(title.iloc[0])}</p>"
        C.value = f"<p>{str(abstract.iloc[0])}</p>"
    else:
        B.value = ""
        C.value = ""

A.observe(update, names="value")

Yes = widgets.Button(value = "Yes", description = "Yes")
Maybe = widgets.Button(value = "Maybe", description = "Maybe")
No = widgets.Button(value = "No", description = "No")

Label = widgets.Label(value = "Result of Title / Abstract screening ")
items = [A, Label, Yes, No]

D = widgets.HBox(items, layout = {"width":"100%"})

screening = display(D, B, C)
update(None)

Note: you may need to restart the kernel to use updated packages.


NameError: name 'csv2df' is not defined

Screening filter #1: Language

In [ ]:
df = pd.read_csv(f"{deduplication}/title+year_deduplicated.csv", encoding = "utf-8")
                
df.to_csv(f"{screening}/screening.csv", encoding = "utf-8")
df.sort_values(by = ['subgroup', 'year', 'authors'])

dict = {"language": "English"}
import pandas as pd
import os
print(df['pmid'])
for x, y in dict.items():
    mask = df[x].astype(str).str.contains(y, case=False, na=False)
    true = df[mask]
    false = df[~mask]
    folder = f"{title_abstract}/{x}"
    os.makedirs(folder, exist_ok = True)
    file = f"{y}"
    true.to_csv(f"{folder}/{file}.csv", encoding = "utf-8")
    false.to_csv(f"{folder}/non_{file}.csv", encoding = "utf-8")
    print(f"Number of records that contain '{y}' in column {x}: ",len(true))
    print(f"Number of records that don't contain '{y}' in column '{x}': ",len(false))


In [ ]:
df.head()

Screening filter #2: Randomized controlled trials

In [ ]:
# Screen for 'randomized' in a newly made 'title and abstract' column
df = true
df['tiab'] = df['title'] + " " + df['abstract']
dict = {"tiab": "random"}
for x, y in dict.items():
    mask = df[x].astype(str).str.contains(y, case=False, na=False)
    true = df[mask]
    false = df[~mask]
    folder = f"{title_abstract}/{x}"
    os.makedirs(folder, exist_ok = True)
    file = f"{y}"
    true.to_csv(f"{folder}/{file}.csv", encoding = "utf-8")
    false.to_csv(f"{folder}/non_{file}.csv", encoding = "utf-8")
    print(f"Number of records that contain '{y}' in column {x}: ",len(true))
    print(f"Number of records that don't contain '{y}' in column '{x}': ",len(false))

Screening filter #3: Clinical trials (unpublished and without results)

In [ ]:
# Screen out unpublished clinical trials from "clinicaltrials.gov" (only published RCTs, from peer-reviewed journals included)
df = true
dict = {"journal": "clinicaltrials.gov"}
for x, y in dict.items():
    mask = df[x].astype(str).str.contains(y, case=False, na=False)
    true = df[mask]
    false = df[~mask]
    folder = f"{title_abstract}/{x}"
    os.makedirs(folder, exist_ok = True)
    file = f"{y}"
    true.to_csv(f"{folder}/{file}.csv", encoding = "utf-8")
    false.to_csv(f"{folder}/non_{file}.csv", encoding = "utf-8")
    print(f"Number of records that contain '{y}' in column {x}: ",len(true))
    print(f"Number of records that don't contain '{y}' in column '{x}': ",len(false))

In [ ]:
df = false
df.to_csv(f"{full_text}/full_text_screening.csv", encoding = "utf-8")

---
[top](#toc) | [search strategy](#search-strategy) | [search](#search) | [deduplication](#deduplication) | [screening](#screening) |
[data collection](#data-collection) |
  
<a id="Full-text screening"></a>

<h3 align="center" style="font-family:Times New Roman;font-variant:small-caps;">Full-text screening</h3>

In [ ]:
import requests
import pandas as pd

df = pd.read_csv(f"{full_text}/full_text_screening.csv", encoding = "utf-8")

doi = df["doi"].fillna("").str.strip()
print(doi)

#sci = df["doi"].where(df["year"]==2010).dropna()

downloads = []
views = []
for x in doi:
    base = f"sci.bban.top/pdf" # options = sci-hub.al
    download = f"https://{base}/{x}.pdf?download=true"
    view = f"https://{base}/{x}.pdf"
    downloads.append(download)
    views.append(view)

df["pdf.download"] = downloads
df["pdf.view"] = views
df.to_csv(f"{screening}/PDF/pdf.csv", encoding = "utf-8")
df.to_csv(f"{screening}/PDF/pdf.csv", encoding = "utf-8")
df.head()
display(df)

---
[top](#toc) | [search strategy](#search-strategy) | [search](#search) | [deduplication](#deduplication) | [screening](#screening) |
[data collection](#data-collection) |
  
<a id="data-collection"></a>

<h1 align="center" style="font-family:Times New Roman;font-variant:small-caps;">Data Collection</h1>

Data collection involves database design and creation, data entry forms design and creation, then data collection, also known as data extraction. 

In [ ]:
df = csv2df("pdf")

In [ ]:
import ipywidgets as widgets
from ipyflex import FlexLayout
from IPython.display import display, Latex, HTML

In [ ]:
df = csv2df("records")
df.head()

<a id="forms"></a>
<div align="center">
<h2 style="font-family:Times New Roman;font-variant: small-caps;">Forms</h2></div>

- [Views](#views)
    - [Layout](#layouts)
    - [Containers](#containers)
    - [Widgets](./src/notebooks/data/components.ipynb)
- [Compose](#compose)
    - [Linking](#linking)
- [Models](#models)

#### Views

<a id="linking"></a>
**Linking** widgets' attributes from the client side

In [ ]:
grid = widgets.Grid()